# Performance Comparison - Library vs FastAPI

Compares calling `airgap_geo` functions **directly** (via `httpx.AsyncClient`) against
calling the same operations through the **FastAPI REST API** (`http://localhost:5050`).

Both paths hit the same Docker backends (Nominatim, Photon, OSRM, postcodes.io), so
this notebook isolates:

1. **FastAPI layer overhead** - the cost of an extra HTTP hop, Pydantic validation, and
   JSON serialisation
2. **Cache benefit** - the API layer's in-process TTL cache (1 hour, 1 024 entries)
3. **Concurrent throughput** - how each path handles parallel load

## Prerequisites

| Service           | Default port | Role                      |
| ----------------- | ------------ | ------------------------- |
| **FastAPI (API)** | 5000         | Unified REST API          |
| Nominatim         | 8080         | Forward geocoding backend |
| Photon            | 2322         | Reverse geocoding backend |
| OSRM + HAProxy    | 80           | Routing backend           |
| postcodes.io      | 8000         | UK postcode backend       |

See **Start Services** below for Docker commands.


______________________________________________________________________

## Start Services

Before running this notebook, make sure the required Docker services are up.
From the **project root**, run:

```bash
make up                  # starts all services (including FastAPI)
```

Check running containers with `make ps`. See the main
[README](../README.md#docker-services) for all available profiles and
configuration details.


## Setup - Imports and Configuration


In [ ]:
import asyncio
import importlib
import statistics
import time

import httpx
import pandas as pd

import airgap_geo.settings as _settings

importlib.reload(_settings)

from airgap_geo import geocoder, lookup_postcode, route
from airgap_geo.settings import API_PORT

client = httpx.AsyncClient()
api_client = httpx.AsyncClient()

API_BASE = f"http://localhost:{API_PORT}"

print("Library client : httpx.AsyncClient (connection-pooled)")
print(f"API base URL   : {API_BASE}")

In [ ]:
async def time_async(coro):
    """Await a coroutine and return (result, elapsed_ms)."""
    t0 = time.perf_counter()
    result = await coro
    elapsed = (time.perf_counter() - t0) * 1000
    return result, elapsed


def time_sync(fn, *args, **kwargs):
    """Call a sync function and return (result, elapsed_ms)."""
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    elapsed = (time.perf_counter() - t0) * 1000
    return result, elapsed


def stats_summary(timings_ms: list[float]) -> dict:
    """Return mean, median, std, min, max for a list of timings."""
    return {
        "mean_ms": round(statistics.mean(timings_ms), 2),
        "median_ms": round(statistics.median(timings_ms), 2),
        "std_ms": round(statistics.stdev(timings_ms), 2)
        if len(timings_ms) > 1
        else 0.0,
        "min_ms": round(min(timings_ms), 2),
        "max_ms": round(max(timings_ms), 2),
    }

______________________________________________________________________

## 1 - Single-Call Latency

Measure the raw overhead of the FastAPI layer by comparing the library (direct) path
against the API (HTTP) path. Each call uses **unique parameters** so the API's TTL
cache never returns a hit - this isolates pure overhead.

We test three services:

| Service   | Library function    | API endpoint         |
| --------- | ------------------- | -------------------- |
| Geocoding | `geocoder()`        | `GET /geocode?q=...` |
| Routing   | `route()`           | `POST /route`        |
| Postcodes | `lookup_postcode()` | `GET /postcodes/...` |


In [ ]:
# --- 1a. Geocoding: 10 unique reverse-geocode queries ----------------

N_CALLS = 10
geocode_queries = [f"{51.5 + i * 0.0001}, {-0.12 + i * 0.0001}" for i in range(N_CALLS)]

lib_geocode_times = []
api_geocode_times = []

for q in geocode_queries:
    _, ms = await time_async(geocoder(q, client))
    lib_geocode_times.append(ms)

for q in geocode_queries:
    _, ms = await time_async(
        api_client.get(f"{API_BASE}/geocode", params={"q": q}, timeout=10)
    )
    api_geocode_times.append(ms)

rows = []
for i in range(N_CALLS):
    rows.append(
        {
            "Call": i + 1,
            "Library (ms)": round(lib_geocode_times[i], 2),
            "API (ms)": round(api_geocode_times[i], 2),
        }
    )

df_geo = pd.DataFrame(rows).set_index("Call")
print("Geocoding - per-call latency (unique queries, no cache hits)\n")
print(df_geo.to_string())
print(f"\nLibrary : {stats_summary(lib_geocode_times)}")
print(f"API     : {stats_summary(api_geocode_times)}")

In [ ]:
# --- 1b. Routing: 10 unique driving routes ---------------------------

ORIGIN = (51.5308, -0.1238)

lib_route_times = []
api_route_times = []

for i in range(N_CALLS):
    dest = (51.49 + i * 0.001, -0.14 + i * 0.001)
    _, ms = await time_async(route(ORIGIN, dest, client, profile="driving"))
    lib_route_times.append(ms)

for i in range(N_CALLS):
    dest_lat = 51.49 + i * 0.001
    dest_lon = -0.14 + i * 0.001
    body = {
        "origin": {"lat": ORIGIN[0], "lon": ORIGIN[1]},
        "destination": {"lat": dest_lat, "lon": dest_lon},
        "profile": "driving",
    }
    _, ms = await time_async(
        api_client.post(f"{API_BASE}/route", json=body, timeout=10)
    )
    api_route_times.append(ms)

rows = []
for i in range(N_CALLS):
    rows.append(
        {
            "Call": i + 1,
            "Library (ms)": round(lib_route_times[i], 2),
            "API (ms)": round(api_route_times[i], 2),
        }
    )

df_rt = pd.DataFrame(rows).set_index("Call")
print("Routing - per-call latency (unique destinations, no cache hits)\n")
print(df_rt.to_string())
print(f"\nLibrary : {stats_summary(lib_route_times)}")
print(f"API     : {stats_summary(api_route_times)}")

In [ ]:
# --- 1c. Postcodes: 10 unique postcode lookups -----------------------

POSTCODES = [
    "SW1A 2AA",
    "EC2V 6DN",
    "WC2N 5DU",
    "SE1 7PB",
    "E1 6AN",
    "W1B 3HH",
    "N1 9GU",
    "NW1 2DB",
    "EC1A 1BB",
    "SW1P 3JA",
]

lib_pc_times = []
api_pc_times = []

for pc in POSTCODES:
    _, ms = await time_async(lookup_postcode(pc, client))
    lib_pc_times.append(ms)

for pc in POSTCODES:
    _, ms = await time_async(api_client.get(f"{API_BASE}/postcodes/{pc}", timeout=10))
    api_pc_times.append(ms)

rows = []
for i, pc in enumerate(POSTCODES):
    rows.append(
        {
            "Postcode": pc,
            "Library (ms)": round(lib_pc_times[i], 2),
            "API (ms)": round(api_pc_times[i], 2),
        }
    )

df_pc = pd.DataFrame(rows).set_index("Postcode")
print("Postcodes - per-call latency (unique postcodes, no cache hits)\n")
print(df_pc.to_string())
print(f"\nLibrary : {stats_summary(lib_pc_times)}")
print(f"API     : {stats_summary(api_pc_times)}")

In [ ]:
# --- 1d. Summary: median latency per service -------------------------

latency_summary = pd.DataFrame(
    [
        {
            "Service": "Geocoding",
            "Library median (ms)": round(statistics.median(lib_geocode_times), 2),
            "API median (ms)": round(statistics.median(api_geocode_times), 2),
        },
        {
            "Service": "Routing",
            "Library median (ms)": round(statistics.median(lib_route_times), 2),
            "API median (ms)": round(statistics.median(api_route_times), 2),
        },
        {
            "Service": "Postcodes",
            "Library median (ms)": round(statistics.median(lib_pc_times), 2),
            "API median (ms)": round(statistics.median(api_pc_times), 2),
        },
    ]
).set_index("Service")

latency_summary["Overhead (ms)"] = (
    latency_summary["API median (ms)"] - latency_summary["Library median (ms)"]
)

print("Single-call latency summary (unique queries, no cache)\n")
latency_summary

______________________________________________________________________

## 2 - Cache Effect

The FastAPI layer wraps each endpoint with a `cachetools.TTLCache` (1 hour TTL,
1 024 entries). The **first** API call for a given query is a cache miss (hits the
backend); the **second** identical call returns instantly from memory.

The library has no built-in caching, so every call always hits the backend.

For each service we compare three timings:

| Path             | Description                           |
| ---------------- | ------------------------------------- |
| **Library**      | Direct call - always hits backend     |
| **API (cold)**   | First API call - cache miss           |
| **API (cached)** | Second identical API call - cache hit |


In [ ]:
# Use fresh queries that haven't been cached by Section 1.

cache_results = []

# --- Geocoding --------------------------------------------------------
GEO_QUERY = "Buckingham Palace, London"

_, lib_geo_ms = await time_async(geocoder(GEO_QUERY, client))
_, api_cold_geo_ms = await time_async(
    api_client.get(f"{API_BASE}/geocode", params={"q": GEO_QUERY}, timeout=10)
)
_, api_warm_geo_ms = await time_async(
    api_client.get(f"{API_BASE}/geocode", params={"q": GEO_QUERY}, timeout=10)
)

cache_results.append(
    {
        "Service": "Geocoding",
        "Library (ms)": round(lib_geo_ms, 2),
        "API cold (ms)": round(api_cold_geo_ms, 2),
        "API cached (ms)": round(api_warm_geo_ms, 2),
    }
)

# --- Routing ----------------------------------------------------------
ROUTE_ORIGIN = (51.5308, -0.1238)
ROUTE_DEST = (51.5007, -0.1246)
ROUTE_BODY = {
    "origin": {"lat": ROUTE_ORIGIN[0], "lon": ROUTE_ORIGIN[1]},
    "destination": {"lat": ROUTE_DEST[0], "lon": ROUTE_DEST[1]},
    "profile": "driving",
}

_, lib_rt_ms = await time_async(
    route(ROUTE_ORIGIN, ROUTE_DEST, client, profile="driving")
)
_, api_cold_rt_ms = await time_async(
    api_client.post(f"{API_BASE}/route", json=ROUTE_BODY, timeout=10)
)
_, api_warm_rt_ms = await time_async(
    api_client.post(f"{API_BASE}/route", json=ROUTE_BODY, timeout=10)
)

cache_results.append(
    {
        "Service": "Routing",
        "Library (ms)": round(lib_rt_ms, 2),
        "API cold (ms)": round(api_cold_rt_ms, 2),
        "API cached (ms)": round(api_warm_rt_ms, 2),
    }
)

# --- Postcodes --------------------------------------------------------
CACHE_PC = "W1A 1AA"

_, lib_pc_ms = await time_async(lookup_postcode(CACHE_PC, client))
_, api_cold_pc_ms = await time_async(
    api_client.get(f"{API_BASE}/postcodes/{CACHE_PC}", timeout=10)
)
_, api_warm_pc_ms = await time_async(
    api_client.get(f"{API_BASE}/postcodes/{CACHE_PC}", timeout=10)
)

cache_results.append(
    {
        "Service": "Postcodes",
        "Library (ms)": round(lib_pc_ms, 2),
        "API cold (ms)": round(api_cold_pc_ms, 2),
        "API cached (ms)": round(api_warm_pc_ms, 2),
    }
)

df_cache = pd.DataFrame(cache_results).set_index("Service")
df_cache["Speedup (cold \u2192 cached)"] = (
    df_cache["API cold (ms)"] / df_cache["API cached (ms)"]
).round(1).astype(str) + "x"

print("Cache effect - single fixed query per service\n")
df_cache

______________________________________________________________________

## 3 - Concurrent Throughput

Fire 20 geocoding requests in parallel via both paths, then compare total
wall-clock time.

Two scenarios:

| Scenario     | Queries                      | API cache behaviour |
| ------------ | ---------------------------- | ------------------- |
| **Unique**   | 20 distinct coordinate pairs | All cache misses    |
| **Repeated** | Same query 20 times          | First miss, 19 hits |

The library has no caching, so both scenarios hit the backend every time. The API
should show a dramatic advantage in the repeated-query scenario.


In [ ]:
# --- 3a. Unique queries (20 distinct coordinates) --------------------

N_CONCURRENT = 20
unique_queries = [
    f"{51.50 + i * 0.001}, {-0.10 + i * 0.001}" for i in range(N_CONCURRENT)
]

# Library: concurrent via asyncio.gather
t0 = time.perf_counter()
await asyncio.gather(*[geocoder(q, client) for q in unique_queries])
lib_unique_total = (time.perf_counter() - t0) * 1000

# API: concurrent via asyncio.gather with httpx.AsyncClient
t0 = time.perf_counter()
await asyncio.gather(
    *[
        api_client.get(f"{API_BASE}/geocode", params={"q": q}, timeout=30)
        for q in unique_queries
    ]
)
api_unique_total = (time.perf_counter() - t0) * 1000

print(f"Concurrent unique queries ({N_CONCURRENT} calls)\n")
print(
    f"  Library  : {lib_unique_total:8.1f} ms total  ({lib_unique_total / N_CONCURRENT:6.1f} ms / call)"
)
print(
    f"  API      : {api_unique_total:8.1f} ms total  ({api_unique_total / N_CONCURRENT:6.1f} ms / call)"
)

In [ ]:
# --- 3b. Repeated queries (same query 20 times) ----------------------

REPEATED_QUERY = "Tower of London"

# Prime the API cache with one call first
await api_client.get(f"{API_BASE}/geocode", params={"q": REPEATED_QUERY}, timeout=10)

# Library: always hits backend
t0 = time.perf_counter()
await asyncio.gather(*[geocoder(REPEATED_QUERY, client) for _ in range(N_CONCURRENT)])
lib_repeat_total = (time.perf_counter() - t0) * 1000

# API: all cache hits after the priming call
t0 = time.perf_counter()
await asyncio.gather(
    *[
        api_client.get(f"{API_BASE}/geocode", params={"q": REPEATED_QUERY}, timeout=10)
        for _ in range(N_CONCURRENT)
    ]
)
api_repeat_total = (time.perf_counter() - t0) * 1000

print(f"Concurrent repeated queries ({N_CONCURRENT} identical calls)\n")
print(
    f"  Library  : {lib_repeat_total:8.1f} ms total  ({lib_repeat_total / N_CONCURRENT:6.1f} ms / call)"
)
print(
    f"  API      : {api_repeat_total:8.1f} ms total  ({api_repeat_total / N_CONCURRENT:6.1f} ms / call)"
)

In [ ]:
# --- 3c. Throughput summary table ------------------------------------

throughput_df = pd.DataFrame(
    [
        {
            "Scenario": f"Unique ({N_CONCURRENT} calls)",
            "Library total (ms)": round(lib_unique_total, 1),
            "API total (ms)": round(api_unique_total, 1),
            "Library per-call (ms)": round(lib_unique_total / N_CONCURRENT, 1),
            "API per-call (ms)": round(api_unique_total / N_CONCURRENT, 1),
        },
        {
            "Scenario": f"Repeated ({N_CONCURRENT} calls)",
            "Library total (ms)": round(lib_repeat_total, 1),
            "API total (ms)": round(api_repeat_total, 1),
            "Library per-call (ms)": round(lib_repeat_total / N_CONCURRENT, 1),
            "API per-call (ms)": round(api_repeat_total / N_CONCURRENT, 1),
        },
    ]
).set_index("Scenario")

print("Concurrent throughput summary\n")
throughput_df

______________________________________________________________________

## 4 - Summary

Headline numbers from all three sections in one table.


In [ ]:
summary_rows = [
    {
        "Metric": "Geocode median latency",
        "Library": f"{statistics.median(lib_geocode_times):.1f} ms",
        "API (cold)": f"{statistics.median(api_geocode_times):.1f} ms",
        "API (cached)": f"{api_warm_geo_ms:.1f} ms",
        "Notes": "Single call, unique vs cached query",
    },
    {
        "Metric": "Route median latency",
        "Library": f"{statistics.median(lib_route_times):.1f} ms",
        "API (cold)": f"{statistics.median(api_route_times):.1f} ms",
        "API (cached)": f"{api_warm_rt_ms:.1f} ms",
        "Notes": "Single call, unique vs cached query",
    },
    {
        "Metric": "Postcode median latency",
        "Library": f"{statistics.median(lib_pc_times):.1f} ms",
        "API (cold)": f"{statistics.median(api_pc_times):.1f} ms",
        "API (cached)": f"{api_warm_pc_ms:.1f} ms",
        "Notes": "Single call, unique vs cached query",
    },
    {
        "Metric": f"Concurrent unique ({N_CONCURRENT}x geocode)",
        "Library": f"{lib_unique_total:.0f} ms",
        "API (cold)": f"{api_unique_total:.0f} ms",
        "API (cached)": "N/A",
        "Notes": "All cache misses",
    },
    {
        "Metric": f"Concurrent repeated ({N_CONCURRENT}x geocode)",
        "Library": f"{lib_repeat_total:.0f} ms",
        "API (cold)": "N/A",
        "API (cached)": f"{api_repeat_total:.0f} ms",
        "Notes": "All cache hits after priming",
    },
]

df_summary = pd.DataFrame(summary_rows).set_index("Metric")
df_summary

### Key takeaways

- **Single-call overhead:** The API adds a small overhead (HTTP hop + Pydantic + cache
  lookup) that is typically dwarfed by backend response times.
- **Cache benefit:** Repeated identical queries via the API return orders of magnitude
  faster than either the library or a cold API call.
- **Concurrent throughput:** Under parallel load with unique queries, both paths are
  roughly comparable (limited by backend concurrency). With repeated queries, the API
  cache provides a dramatic throughput advantage.


______________________________________________________________________

## Teardown - Close HTTP Clients


In [ ]:
await client.aclose()
await api_client.aclose()

______________________________________________________________________

## Stop Services

To stop containers, use `make down` from the project root. See the
[README](../README.md#docker-services) for details.
